# Step 6 — Experiment Runner

**Thesis:** Drift-Aware Selective Updating of Two-Stage Tabular ML Pipelines  
**Goal:** Run the full end-to-end experiment loop across a grid of configurations, compare selective retraining strategies, and produce the final results CSV for thesis analysis.

Experiment grid:
- **Datasets:** `adult`, `diabetes_readmission`
- **Models:** `logreg`, `xgboost`
- **Drift types × severities:** `covariate/concept/both` × `low/medium/high`
- **Retraining strategies:** no retraining, Stage 1 only, Stage 2 only, full retraining

Reference implementation: `drift_framework/run_experiment.py`

## 6.1 Setup

In [ ]:
import sys, os, copy
from pathlib import Path

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

from drift_framework.data.loader import load_dataset, DataBundle
from drift_framework.pipeline.two_stage import TwoStagePipeline
from drift_framework.drift.injector import DriftInjector
from drift_framework.metrics.tracker import MetricsTracker, timed_ram_block
from drift_framework.monitoring.evidently_monitor import EvidentlyMonitor
from drift_framework.monitoring.river_monitor import RiverMonitor

SEED = 42
RESULTS_DIR = Path(PROJECT_ROOT) / "results"
RESULTS_DIR.mkdir(exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "experiment_results.csv"

print("Setup OK")

## 6.2 Experiment configuration

In [ ]:
DATASETS = ["adult"]         # add "diabetes_readmission" once tableshift is set up
MODELS   = ["logreg", "xgboost"]
DRIFT_TYPES = ["covariate", "concept", "both"]
SEVERITIES  = ["low", "medium", "high"]
START_IDX   = 0

total_runs = len(DATASETS) * len(MODELS) * len(DRIFT_TYPES) * len(SEVERITIES)
print(f"Total experiment runs: {total_runs}")

## 6.3 Single-run function

In [ ]:
def run_single_experiment(
    bundle: DataBundle,
    model_type: str,
    drift_type: str,
    severity: str,
    start_idx: int = 0,
) -> dict:
    """
    One full experiment run:
      1. Train baseline pipeline on reference split
      2. Evaluate on pre-drift split (baseline AUC)
      3. Inject drift into post-drift split
      4. Evaluate on drifted data (no retraining)
      5. Evaluate after Stage-1-only retraining
      6. Evaluate after Stage-2-only retraining
      7. Evaluate after full retraining
      8. Run Evidently batch detection
      9. Run River ADWIN online detection
    Returns a dict of all metrics.
    """
    # ---- 1. Train baseline ----
    pipe = TwoStagePipeline(
        model_type=model_type,
        num_features=bundle.num_features,
        cat_features=bundle.cat_features,
    )
    with timed_ram_block() as train_m:
        pipe.fit(bundle.X_ref, bundle.y_ref)

    # ---- 2. Baseline AUC ----
    proba_pre = pipe.predict_proba(bundle.X_pre)[:, 1]
    auc_baseline = roc_auc_score(bundle.y_pre, proba_pre)

    # ---- 3. Inject drift ----
    injector = DriftInjector(drift_type=drift_type, severity=severity, start_idx=start_idx)
    injector.fit(bundle.X_ref, bundle.num_features)
    X_post_d, y_post_d = injector.inject(bundle.X_post, bundle.y_post)

    # ---- 4. Evaluate without retraining ----
    proba_no_retrain = pipe.predict_proba(X_post_d)[:, 1]
    auc_no_retrain = roc_auc_score(y_post_d, proba_no_retrain)

    # ---- 5. Stage-1-only retraining (preprocessor adapts to new distribution) ----
    pipe_s1 = copy.deepcopy(pipe)
    with timed_ram_block() as s1_m:
        pipe_s1.fit_stage1(X_post_d, y_post_d)
    proba_s1 = pipe_s1.predict_proba(X_post_d)[:, 1]
    auc_s1 = roc_auc_score(y_post_d, proba_s1)

    # ---- 6. Stage-2-only retraining (model sees new labels, preprocessor frozen) ----
    pipe_s2 = copy.deepcopy(pipe)
    with timed_ram_block() as s2_m:
        pipe_s2.fit_stage2(X_post_d, y_post_d)
    proba_s2 = pipe_s2.predict_proba(X_post_d)[:, 1]
    auc_s2 = roc_auc_score(y_post_d, proba_s2)

    # ---- 7. Full retraining ----
    pipe_full = TwoStagePipeline(
        model_type=model_type,
        num_features=bundle.num_features,
        cat_features=bundle.cat_features,
    )
    with timed_ram_block() as full_m:
        pipe_full.fit(X_post_d, y_post_d)
    proba_full = pipe_full.predict_proba(X_post_d)[:, 1]
    auc_full = roc_auc_score(y_post_d, proba_full)

    # ---- 8. Evidently batch detection ----
    ev_monitor = EvidentlyMonitor(
        num_features=bundle.num_features,
        cat_features=bundle.cat_features,
    )
    ev_result = ev_monitor.detect(bundle.X_ref, X_post_d)

    # ---- 9. River ADWIN online detection ----
    river_monitor = RiverMonitor(delta=0.002)
    river_result = river_monitor.detect(X_post_d, y_post_d, predict_fn=pipe.predict)

    return {
        "dataset": bundle.name,
        "model": model_type,
        "drift_type": drift_type,
        "severity": severity,
        "start_idx": start_idx,
        # Performance
        "auc_baseline": auc_baseline,
        "auc_no_retrain": auc_no_retrain,
        "auc_stage1_only": auc_s1,
        "auc_stage2_only": auc_s2,
        "auc_full_retrain": auc_full,
        "delta_auc_no_retrain": auc_baseline - auc_no_retrain,
        "delta_auc_stage1": auc_baseline - auc_s1,
        "delta_auc_stage2": auc_baseline - auc_s2,
        "delta_auc_full": auc_baseline - auc_full,
        # Compute
        "train_time_sec": train_m["elapsed_sec"],
        "train_ram_mb": train_m["peak_ram_mb"],
        "s1_retrain_time_sec": s1_m["elapsed_sec"],
        "s2_retrain_time_sec": s2_m["elapsed_sec"],
        "full_retrain_time_sec": full_m["elapsed_sec"],
        # Detection
        "evidently_dataset_drift": ev_result["dataset_drift"],
        "evidently_n_drifted": ev_result["n_drifted_features"],
        "river_first_drift_index": river_result["first_drift_index"],
        "river_n_detections": river_result["n_drift_detections"],
    }

## 6.4 Load all datasets

In [ ]:
# Load datasets once and cache in memory
dataset_bundles = {}
for ds_name in DATASETS:
    dataset_bundles[ds_name] = load_dataset(ds_name)

print("Datasets loaded:", list(dataset_bundles.keys()))

## 6.5 Run the experiment grid

> **Note:** This cell runs all combinations and can take 10–30 minutes depending on your hardware. For a quick test, reduce `MODELS` to `["logreg"]` and `SEVERITIES` to `["high"]`.

In [ ]:
all_results = []

for ds_name in DATASETS:
    bundle = dataset_bundles[ds_name]
    for model_type in MODELS:
        for drift_type in DRIFT_TYPES:
            for severity in SEVERITIES:
                print(f"\n{'─'*60}")
                print(f"dataset={ds_name}  model={model_type}  drift={drift_type}/{severity}")
                print(f"{'─'*60}")
                try:
                    result = run_single_experiment(
                        bundle=bundle,
                        model_type=model_type,
                        drift_type=drift_type,
                        severity=severity,
                        start_idx=START_IDX,
                    )
                    all_results.append(result)
                    print(
                        f"  AUC: baseline={result['auc_baseline']:.4f}  "
                        f"no-retrain={result['auc_no_retrain']:.4f}  "
                        f"s1-only={result['auc_stage1_only']:.4f}  "
                        f"s2-only={result['auc_stage2_only']:.4f}  "
                        f"full={result['auc_full_retrain']:.4f}"
                    )
                except Exception as e:
                    print(f"  ERROR: {e}")

print(f"\nCompleted {len(all_results)} / {total_runs} runs.")

## 6.6 Save results to CSV

In [ ]:
results_df = pd.DataFrame(all_results)

# Append to CSV (write header only on first run)
write_header = not RESULTS_CSV.exists()
results_df.to_csv(RESULTS_CSV, mode="a", header=write_header, index=False)
print(f"Results saved to {RESULTS_CSV}  ({len(results_df)} rows)")

results_df.head()

## 6.7 Analysis: ΔAUC by drift type and severity

In [ ]:
# Load the full CSV (may include previous runs)
df = pd.read_csv(RESULTS_CSV)
print(f"Total rows in CSV: {len(df)}")
df.describe()

In [ ]:
# ΔAUC by drift type (no retraining vs. full retraining)
pivot = df.groupby(["drift_type", "severity"])[["delta_auc_no_retrain", "delta_auc_full"]].mean().reset_index()
pivot

In [ ]:
# Grouped bar chart: ΔAUC (no retraining) by drift type and severity
severity_order = ["low", "medium", "high"]
drift_types = df["drift_type"].unique()

fig, axes = plt.subplots(1, len(drift_types), figsize=(14, 5), sharey=True)

for ax, dt in zip(axes, drift_types):
    subset = df[df["drift_type"] == dt]
    sev_means = subset.groupby("severity")["delta_auc_no_retrain"].mean()
    sev_means = sev_means.reindex(severity_order)
    ax.bar(sev_means.index, sev_means.values, color=["#4CAF50", "#FFC107", "#F44336"])
    ax.set_title(f"Drift: {dt}")
    ax.set_xlabel("Severity")
    if ax == axes[0]:
        ax.set_ylabel("Mean ΔAUC (no retraining)")
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")

plt.suptitle("AUC degradation by drift type and severity (positive = performance drop)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# Compare retraining strategies
strategy_cols = {
    "No retraining": "delta_auc_no_retrain",
    "Stage 1 only":  "delta_auc_stage1",
    "Stage 2 only":  "delta_auc_stage2",
    "Full retrain":  "delta_auc_full",
}

strategy_means = {
    label: df[col].mean()
    for label, col in strategy_cols.items()
}

fig, ax = plt.subplots(figsize=(8, 4))
labels = list(strategy_means.keys())
values = list(strategy_means.values())
colors = ["#F44336", "#2196F3", "#FF9800", "#4CAF50"]
ax.bar(labels, values, color=colors)
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_ylabel("Mean ΔAUC (averaged over all drift configs)")
ax.set_title("Retraining strategy comparison (lower ΔAUC = better recovery)")
plt.tight_layout()
plt.show()

for label, val in strategy_means.items():
    print(f"  {label:20s}: mean ΔAUC = {val:+.4f}")

## 6.8 Detection timing: Evidently vs River

In [ ]:
print("Detection summary (mean over all runs):")
print(f"  Evidently detected drift in: {df['evidently_dataset_drift'].sum()} / {len(df)} runs")
print(f"  River ADWIN mean first detection index: {df['river_first_drift_index'].dropna().mean():.0f}")
print(f"  River ADWIN mean detections per run: {df['river_n_detections'].mean():.1f}")

## 6.9 Sanity checks

In [ ]:
df = pd.read_csv(RESULTS_CSV)

# 1. CSV has all expected columns
required_cols = [
    "dataset", "model", "drift_type", "severity", "start_idx",
    "auc_baseline", "auc_no_retrain", "auc_stage1_only", "auc_stage2_only", "auc_full_retrain",
    "delta_auc_no_retrain", "evidently_dataset_drift", "river_first_drift_index",
]
for col in required_cols:
    assert col in df.columns, f"Missing column: {col}"
print("CSV has all required columns — OK")

# 2. No all-NaN rows
assert not df.isnull().all(axis=1).any(), "Found an all-NaN row in results CSV"
print("No all-NaN rows in CSV — OK")

# 3. AUC values are in [0, 1]
for col in ["auc_baseline", "auc_no_retrain", "auc_stage1_only", "auc_stage2_only", "auc_full_retrain"]:
    assert df[col].between(0, 1).all(), f"{col} has values outside [0,1]"
print("All AUC values in [0, 1] — OK")

# 4. High severity should produce larger ΔAUC than low severity (on average)
for dt in df["drift_type"].unique():
    low_mean  = df[(df["drift_type"] == dt) & (df["severity"] == "low" )]["delta_auc_no_retrain"].mean()
    high_mean = df[(df["drift_type"] == dt) & (df["severity"] == "high")]["delta_auc_no_retrain"].mean()
    if not np.isnan(low_mean) and not np.isnan(high_mean):
        assert high_mean >= low_mean, \
            f"{dt}: high severity ΔAUC ({high_mean:.4f}) < low severity ({low_mean:.4f})"
print("High severity → larger ΔAUC than low severity — OK")

print(f"\nAll sanity checks passed! ({len(df)} experiment rows in {RESULTS_CSV})")